In [53]:
# PRACTICE GETTING AGENTS TO USE TOOLS
from langchain_ollama import ChatOllama
from tools import *

accession='AAL87837'

tools={
    'FetchNcbiMetadata': FetchNcbiMetadata,
    'FetchLiterature': FetchLiterature
}

# Initialise model
llm=ChatOllama(
    model="qwen3.5",
    temperature=0
)

# Give model tools to use. (This case isnt required but may be needed for later when decisions need to be made)
llm=llm.bind_tools(tools.values())

response=llm.invoke(f'What is the literature associated for this ncbi accession {accession}')

print(response)
# Have to manually call the functions, parsing the function and the

content='' additional_kwargs={} response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-21T20:10:33.386871Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4980523541, 'load_duration': 102422250, 'prompt_eval_count': 383, 'prompt_eval_duration': 1084472875, 'eval_count': 101, 'eval_duration': 3757105542, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'} id='lc_run--019db1aa-68b5-7233-b787-9ccf80415a3d-0' tool_calls=[{'name': 'FetchLiterature', 'args': {'accession': 'AAL87837'}, 'id': '23555c87-02ad-40cb-bdaa-571c657370ef', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 383, 'output_tokens': 101, 'total_tokens': 484}


In [54]:
# Execute function using args provided by model

# Iterate over tool calls
for tool_call in response.tool_calls:
    tool_name = tool_call["name"] # get the function to use
    args = tool_call["args"] # get the args
    result = tools[tool_name](**args)

    print(f'############### RESULT ###############\n{result}\n\n\n')


############### RESULT ###############
{'Snapshot of the genome of the pseudo-T-even bacteriophage RB49.': '<?xml version="1.0" ?>\n<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n<PubmedArticleSet>\n<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" IndexingMethod="Manual"><PMID Version="1">11976309</PMID><DateCompleted><Year>2002</Year><Month>05</Month><Day>22</Day></DateCompleted><DateRevised><Year>2021</Year><Month>05</Month><Day>26</Day></DateRevised><Article PubModel="Print"><Journal><ISSN IssnType="Print">0021-9193</ISSN><JournalIssue CitedMedium="Print"><Volume>184</Volume><Issue>10</Issue><PubDate><Year>2002</Year><Month>May</Month></PubDate></JournalIssue><Title>Journal of bacteriology</Title><ISOAbbreviation>J Bacteriol</ISOAbbreviation></Journal><ArticleTitle>Snapshot of the genome of the pseudo-T-even bacteriophage RB49.</ArticleTitle><Pagination><StartPage>2789</Sta

In [55]:
# model prompts to analyse papers
summary=llm.invoke(f'From the dictionaries of papers provided please tell me what the host organism of this phage is. {result}')

final_species_output=llm.invoke(f'From this summary please respond with only the name of the host organism. {summary}')


In [56]:
print(f'First model call\n{summary.content}\n\nSecond model call\n{final_species_output.content}')


First model call
Based on the literature provided, the host organism of bacteriophage RB49 is **Escherichia coli (E. coli)**.

This is explicitly stated in the first paper's abstract: "RB49 is a virulent bacteriophage that infects Escherichia coli." The second paper further confirms this by discussing how RB49-encoded proteins can substitute for E. coli chaperonins, indicating that RB49 infects and replicates within E. coli cells.

Second model call
Escherichia coli


In [ ]:
# Now i have pipeline to go from accession to the host species. Duration is roughly 15 seconds